### **Privacy & Data Minimization**

| Component | What it does | Practical example | Key limitation or consideration |
| --- | --- | --- | --- |
| Purpose limitation | Defines why data is used and which fields exist | Identify that email is not needed for issue counts | Purpose can change; reassess before reuse |
| Data Minimization | Retains only task-relevant fields | Keep only `support_issue` and `plan` | A field that seems harmless may become identifying in combination |
| Sensitivity inspection | Flags direct identifiers and uncertain fields | Detect an email address using its value pattern | Heuristics can miss sensitive data or misclassify a field |
| Masking | Hides part or all of a displayed value | Show `a***@example.invalid` | Remaining fragments may still identify someone |
| Pseudonymization | Replaces identifiers with stable tokens | Replace an email with `PERSON-014` | A key or linkage may reconnect the token to a person |
| Generalization | Replaces precise values with broader categories | Convert exact ages into age ranges | Other fields may still identify someone when combined |
| Aggregation | Reports group-level statistics instead of individual rows | Publish average resolution time by issue type | Small or overlapping groups can leak information |
| Suppression | Hides values or groups that are too small | Publish issue counts only for groups of five or more | Choose thresholds; suppression alone does not guarantee anonymity |

In [1]:
#Import Libs
import hashlib
import hmac
import re

In [2]:
#Records Example
records = [
    {"name": "Avery Example", "email": "avery@example.invalid", "age": 34, "city": "Harborview", "plan": "basic", "support_issue": "password reset"},
    {"name": "Blake Sample", "email": "blake@example.invalid", "age": 42, "city": "Pine Ridge", "plan": "pro", "support_issue": "billing question"},
    {"name": "Casey Demo", "email": "casey@example.invalid", "age": 29, "city": "Harborview", "plan": "basic", "support_issue": "password reset"},
]

### **01. Minimization Task**

In [3]:
#Define the Key Terms & Analytics Field
SENSITIVE_KEY_TERMS = {
    "name", "email", "phone", "mobile", "address", "street", "postcode", "zip",
    "birth", "dob", "ssn", "tax_id", "passport", "password", "token", "secret",
    "account_number", "customer_id", "person_id", "ip_address", "location",
}
KNOWN_ANALYTICS_FIELDS = {
    "support_issue", "issue", "plan", "minutes", "resolution_minutes", "category", "status"
}

In [4]:
#Create the Patterns
EMAIL_PATTERN = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
PHONE_PATTERN = re.compile(r"^\+?[0-9][0-9 .()\-]{7,}$")

In [5]:
#Function to Normalize Data
def normalize_records(data):
    if isinstance(data, dict):
        return [data], "single_record"
    if isinstance(data, list) and all(isinstance(row, dict) for row in data):
        return data, "list_of_records"
    raise TypeError("Expected a dictionary or a list of dictionaries.")

In [6]:
#Function to Find Sensitive Terms
def looks_sensitive(key, value):
    key_lower = key.lower()
    if any(term in key_lower for term in SENSITIVE_KEY_TERMS):
        return True
    if isinstance(value, str) and (EMAIL_PATTERN.match(value.strip()) or PHONE_PATTERN.match(value.strip())):
        return True
    return key_lower not in KNOWN_ANALYTICS_FIELDS

In [7]:
#Function to Inspect and Minimize
def inspect_and_minimize(data, selected_fields):
    rows, input_format = normalize_records(data)
    all_fields = sorted({key for row in rows for key in row})
    field_is_sensitive = {
        field: any(looks_sensitive(field, row.get(field)) for row in rows if field in row)
        for field in all_fields
    }
    recommended_fields = [field for field in all_fields if not field_is_sensitive[field]]
    accepted_fields = [field for field in selected_fields if field in recommended_fields]

    selected_dataset = [
        {field: row[field] for field in accepted_fields if field in row}
        for row in rows
    ]

    marked_dataset = [
        {
            field: ("Minimized" if field_is_sensitive[field] else value)
            for field, value in row.items()
        }
        for row in rows
    ]
    report = {
        "input_format": input_format,
        "fields_found": all_fields,
        "recommended_fields": recommended_fields,
        "sensitive_or_uncertain_fields": [field for field in all_fields if field_is_sensitive[field]],
        "selected_fields_used": accepted_fields,
        "selected_fields_skipped": [field for field in selected_fields if field not in accepted_fields],
    }
    return report, selected_dataset, marked_dataset

In [8]:
#Application Example
report, selected_dataset, marked_dataset = inspect_and_minimize(records,selected_fields=["support_issue", "plan"],)

In [9]:
#Show the Results
report

{'input_format': 'list_of_records',
 'fields_found': ['age', 'city', 'email', 'name', 'plan', 'support_issue'],
 'recommended_fields': ['plan', 'support_issue'],
 'sensitive_or_uncertain_fields': ['age', 'city', 'email', 'name'],
 'selected_fields_used': ['support_issue', 'plan'],
 'selected_fields_skipped': []}

In [10]:
#Choose only the Recommended Fields
MINIMIZATION_FIELDS = ["support_issue", "plan"]
minimized_records = [
    {field: row[field] for field in MINIMIZATION_FIELDS
     if field in row and field in report["recommended_fields"]}
    for row in records
]

In [11]:
#Minimize the Records
#Show the Report
minimized_records_with_markers = [
    {
        field: ("Minimized" if looks_sensitive(field, value) else value)
        for field, value in row.items()
    }
    for row in records
]

In [12]:
#Show the Results
print("Full minimized dataset (selected fields only):")
for row in minimized_records:
    print(row)

print("\nFull dataset with sensitive or uncertain values replaced:")
for row in minimized_records_with_markers:
    print(row)

Full minimized dataset (selected fields only):
{'support_issue': 'password reset', 'plan': 'basic'}
{'support_issue': 'billing question', 'plan': 'pro'}
{'support_issue': 'password reset', 'plan': 'basic'}

Full dataset with sensitive or uncertain values replaced:
{'name': 'Minimized', 'email': 'Minimized', 'age': 'Minimized', 'city': 'Minimized', 'plan': 'basic', 'support_issue': 'password reset'}
{'name': 'Minimized', 'email': 'Minimized', 'age': 'Minimized', 'city': 'Minimized', 'plan': 'pro', 'support_issue': 'billing question'}
{'name': 'Minimized', 'email': 'Minimized', 'age': 'Minimized', 'city': 'Minimized', 'plan': 'basic', 'support_issue': 'password reset'}


### **02. Masking and Pseudonymization Task**

In [13]:
#Define a Key
DEMO_PSEUDONYM_KEY = b"not-a-production-key"

In [14]:
#Function to Create a Mask Value
def mask_value(field, value):
    if value is None:
        return None
    text = str(value)
    field_lower = field.lower()
    if "email" in field_lower and "@" in text:
        local, domain = text.split("@", 1)
        visible = local[:1] if local else "*"
        return f"{visible}***@{domain}"
    if "phone" in field_lower or "mobile" in field_lower:
        digits = "".join(char for char in text if char.isdigit())
        return f"***{digits[-2:]}" if len(digits) >= 2 else "Minimized"
    if "name" in field_lower:
        parts = text.split()
        return " ".join(part[0] + "***" for part in parts if part)
    if looks_sensitive(field, value):
        return "Minimized"
    return value

In [15]:
#Function to Pseudonym
def pseudonym_for(field, value):
    # Stable, field-scoped token for this demonstration; keep the key secret.
    message = f"{field.lower()}|{value}".encode("utf-8")
    digest = hmac.new(DEMO_PSEUDONYM_KEY, message, hashlib.sha256).hexdigest()[:10]
    return f"{field.upper()}-{digest}"

In [16]:
#Apply the Masking and Pseudonymization
masked_records = [
    {field: mask_value(field, value) for field, value in row.items()}
    for row in records
]

pseudonymized_records = [
    {
        field: (pseudonym_for(field, value) if looks_sensitive(field, value) else value)
        for field, value in row.items()
    }
    for row in records
]

In [17]:
#Show the Results
print("Complete masked dataset:")
for row in masked_records:
    print(row)

print("\nComplete pseudonymized dataset:")
for row in pseudonymized_records:
    print(row)

Complete masked dataset:
{'name': 'A*** E***', 'email': 'a***@example.invalid', 'age': 'Minimized', 'city': 'Minimized', 'plan': 'basic', 'support_issue': 'password reset'}
{'name': 'B*** S***', 'email': 'b***@example.invalid', 'age': 'Minimized', 'city': 'Minimized', 'plan': 'pro', 'support_issue': 'billing question'}
{'name': 'C*** D***', 'email': 'c***@example.invalid', 'age': 'Minimized', 'city': 'Minimized', 'plan': 'basic', 'support_issue': 'password reset'}

Complete pseudonymized dataset:
{'name': 'NAME-58d9fcc231', 'email': 'EMAIL-1ea0820d27', 'age': 'AGE-172100c735', 'city': 'CITY-716a357a66', 'plan': 'basic', 'support_issue': 'password reset'}
{'name': 'NAME-36a7668c03', 'email': 'EMAIL-9397a3a5ca', 'age': 'AGE-518987cfe8', 'city': 'CITY-f81173fb62', 'plan': 'pro', 'support_issue': 'billing question'}
{'name': 'NAME-082f424ea5', 'email': 'EMAIL-36595b661e', 'age': 'AGE-e0346e02ad', 'city': 'CITY-716a357a66', 'plan': 'basic', 'support_issue': 'password reset'}


### **03. Generalization, Aggregation, and k-Anonymity**

Removing names and e-mails is not enough: a combination of ordinary fields (age, city, plan) can still single out a person. **k-anonymity** requires every combination of these *quasi-identifiers* to be shared by at least *k* records. Groups smaller than *k* are suppressed or generalized further.

In [18]:
#Create a Larger Synthetic Dataset (no direct identifiers)
support_tickets = [
    {"age": 23, "city": "Harborview", "plan": "basic", "issue": "password reset"},
    {"age": 27, "city": "Harborview", "plan": "basic", "issue": "billing question"},
    {"age": 29, "city": "Harborview", "plan": "pro", "issue": "password reset"},
    {"age": 34, "city": "Harborview", "plan": "basic", "issue": "password reset"},
    {"age": 36, "city": "Harborview", "plan": "pro", "issue": "billing question"},
    {"age": 38, "city": "Harborview", "plan": "basic", "issue": "app error"},
    {"age": 41, "city": "Pine Ridge", "plan": "pro", "issue": "billing question"},
    {"age": 44, "city": "Pine Ridge", "plan": "pro", "issue": "app error"},
    {"age": 47, "city": "Pine Ridge", "plan": "basic", "issue": "password reset"},
    {"age": 71, "city": "Pine Ridge", "plan": "pro", "issue": "billing question"},
]

#Function to Generalize Age into Ranges
def age_range(age, width=10):
    start = (age // width) * width
    return f"{start}-{start + width - 1}"

generalized = [{**row, "age": age_range(row["age"])} for row in support_tickets]
for row in generalized[:3]:
    print(row)

{'age': '20-29', 'city': 'Harborview', 'plan': 'basic', 'issue': 'password reset'}
{'age': '20-29', 'city': 'Harborview', 'plan': 'basic', 'issue': 'billing question'}
{'age': '20-29', 'city': 'Harborview', 'plan': 'pro', 'issue': 'password reset'}


In [19]:
#Function to Check k-Anonymity and Suppress Small Groups
from collections import Counter

def k_anonymity_report(rows, quasi_identifiers, k=3):
    groups = Counter(tuple(row[q] for q in quasi_identifiers) for row in rows)
    small = {group: size for group, size in groups.items() if size < k}
    released = [row for row in rows if groups[tuple(row[q] for q in quasi_identifiers)] >= k]
    return groups, small, released

groups, small, released = k_anonymity_report(generalized, ["age", "city"], k=3)
print("Group sizes:", dict(groups))
print("Groups below k=3 (suppressed):", small)
print("Records released:", len(released), "of", len(generalized))

Group sizes: {('20-29', 'Harborview'): 3, ('30-39', 'Harborview'): 3, ('40-49', 'Pine Ridge'): 3, ('70-79', 'Pine Ridge'): 1}
Groups below k=3 (suppressed): {('70-79', 'Pine Ridge'): 1}
Records released: 9 of 10


In [20]:
#Aggregate Instead of Releasing Rows (with small-count suppression)
def aggregate(rows, field, min_count=3):
    counts = Counter(row[field] for row in rows)
    return {value: (count if count >= min_count else f"<{min_count}") for value, count in counts.items()}

print("Tickets by issue:", aggregate(support_tickets, "issue"))
print("Tickets by plan:", aggregate(support_tickets, "plan"))

Tickets by issue: {'password reset': 4, 'billing question': 4, 'app error': '<3'}
Tickets by plan: {'basic': 5, 'pro': 5}


### **Key Takeaways**

* Minimize first: the safest field is the one you never collect or send.
* Masking and pseudonymization reduce exposure, but pseudonymized data is still personal data under the LGPD and the GDPR.
* Check combinations of ordinary fields with k-anonymity, and prefer aggregated outputs with small-count suppression when publishing statistics.